In [99]:
import numpy as np
from tqdm import tqdm

rng = np.random.default_rng()

Субъект (особь) - набор индексов разработчиков, назначенных на каждую задачу по порядку. (по факту: особь - ответ)

In [3]:
CATEGORIES_COUNT = 4

In [4]:
with open("input.txt") as f:
    tasks_count = N = np.loadtxt(f, max_rows=1, dtype=np.int64)
    tasks_categories = np.loadtxt(f, max_rows=1, dtype=np.int64) - 1  # (n,)
    tasks_times = np.loadtxt(f, max_rows=1, dtype=np.float64)  # (n,)
    developers_count = M = np.loadtxt(f, max_rows=1, dtype=np.int64)
    developers_coefficients = np.loadtxt(f, max_rows=M, dtype=np.float64)  # (n, m)

In [5]:
def create_random_population(genes_count: int, subjects_count: int) -> np.ndarray:
    return rng.integers(0, developers_count, size=(subjects_count, genes_count))

In [60]:
I_categories = np.eye(CATEGORIES_COUNT, dtype=bool)

def get_fitness_minimal(population: np.ndarray) -> np.ndarray:
    assert population.ndim == 2

    times = developers_coefficients[population][..., I_categories[tasks_categories]]
    times *= tasks_times

    mask = population == np.arange(developers_count)[:, None, None]
    masked = np.where(mask, times, 0)
    return masked.sum(axis=-1).max(axis=0)

def get_fitness_answer(population: np.ndarray) -> np.ndarray:
    fit = get_fitness_minimal(population)
    return 100 / (1e-10 + fit)

def get_fitness(population: np.ndarray) -> np.ndarray:
    fit = get_fitness_minimal(population)
    return np.exp(590-fit)

In [7]:
def su_sampling(fitness: np.ndarray, n: int, start: float = None) -> np.ndarray:
    _fitness = fitness.copy()
    _fitness.sort()
    _fitness = _fitness[::-1]
    argsort = fitness.argsort()[::-1]

    fitness_cumsum = np.cumsum(_fitness)
    h = fitness_cumsum[-1] / n

    if start is None:
        start = rng.uniform(0, h)

    pointers = np.arange(start, start + (n - 0.5) * h, h)
    mask = pointers[:, None] < fitness_cumsum
    return argsort[mask.argmax(axis=-1)]

In [8]:
def one_point_crossover(population_l: np.ndarray, population_r: np.ndarray, point: np.ndarray) -> np.ndarray:
    mask = np.arange(population_l.shape[1]) < point[:, None]
    return np.where(mask, population_l, population_r)

In [9]:
def create_new_population(old_population: np.ndarray, sampled_idx: np.ndarray, count: int) -> np.ndarray:
    sampled_population = old_population[sampled_idx]

    subject1_idx = rng.choice(sampled_population.shape[0], count)
    subject2_idx = (subject1_idx + rng.integers(1, sampled_population.shape[0], count)) % sampled_population.shape[0]

    points = rng.choice(sampled_population.shape[1], count)

    new_population = one_point_crossover(sampled_population[subject1_idx], sampled_population[subject2_idx], points)

    return new_population

In [10]:
def create_mutations(population: np.ndarray, gen_mutation_chance: float = 0.01) -> np.ndarray:
    gen_mutation_chance = gen_mutation_chance or 0.01
    new_population = population.copy()

    mutated_mask = rng.random(new_population.shape) < gen_mutation_chance
    total_mutated = mutated_mask.sum()

    new_population[mutated_mask] += rng.integers(1, developers_count, total_mutated)
    new_population[mutated_mask] %= developers_count
    return new_population

### !

In [13]:
def genetic_iterations(start_population: np.ndarray, iterations_number: int, mutations_chance: float = None) -> np.ndarray:
    _population = start_population
    for _ in range(iterations_number):
        fitness = get_fitness(_population)
        sampled_subjects_idx = su_sampling(fitness, min(POPULATION_COUNT//20, 10))
        new_population = create_new_population(_population, sampled_subjects_idx, POPULATION_COUNT)
        _population = create_mutations(new_population, mutations_chance)

    return _population

In [15]:
def save_data(_population: np.ndarray):
    _fit = get_fitness(_population)
    _fit_ans = get_fitness_answer(_population)
    _fit_min = get_fitness_minimal(_population)

    _arg = _fit.argmax()
    _max = _fit.max()
    _ans = _fit_ans.max()
    _min = _fit_min.min()
    print(_ans, _max, _min, _arg)
    with open(f"output_info.txt", "a") as fi:
        print(_ans, _max, _min, _arg, file=fi)
        print(*(_population[_arg] + 1), file=fi)

In [84]:
POPULATION_COUNT = 100

In [213]:
from io import StringIO
# found_min = "1 " * int(tasks_count)
found_min = ""
found_min = np.loadtxt(StringIO(found_min), dtype=int)
# print(*found_min)
# found_min

In [214]:
# pop = found_min[None].repeat(POPULATION_COUNT, axis=0) - 1

In [249]:
# pop[1:] = create_mutations(pop[1:])

In [85]:
pop = create_random_population(tasks_count, POPULATION_COUNT)
pop

array([[4, 7, 7, ..., 6, 3, 7],
       [3, 4, 5, ..., 5, 4, 0],
       [2, 8, 0, ..., 1, 1, 9],
       ...,
       [6, 2, 3, ..., 6, 6, 9],
       [4, 4, 5, ..., 1, 9, 2],
       [6, 0, 9, ..., 8, 9, 3]], shape=(100, 1000))

In [ ]:
get_fitness(pop)

In [87]:
running_max_pop = pop[0]
running_max_fit = get_fitness_answer(pop)[0]

In [102]:
for _j in tqdm(range(10000)):
    pop = genetic_iterations(pop, 1, mutations_chance=0.01)

    fitness = get_fitness_answer(pop)
    _argmax = fitness.argmax()
    if fitness[_argmax] > running_max_fit:
        running_max_pop = pop[_argmax]
        running_max_fit = fitness[_argmax]
        save_data(pop)
    else:
        with open("output_log.txt", "a") as f_log:
            print(fitness[_argmax], file=f_log)

 60%|█████▉    | 5959/10000 [01:25<00:59, 67.76it/s]

0.16677924265543126 6.80682281952933e-05 599.595 27


100%|██████████| 10000/10000 [02:26<00:00, 68.45it/s]
